# 04 · Explicabilidad fuera de línea (D-13)

Grad-CAM, estabilidad de las explicaciones, relevancia frente a lesiones anotadas, atención del modelo H2 y SHAP
(requiere `pip install -e ".[explain]"`). SHAP es costoso: use una muestra pequeña.

In [ ]:
import mlflow
import numpy as np
import torch
from PIL import Image

from potato_blight.explain.gradcam import GradCAM
from potato_blight.explain.quality import relevance, stability
from potato_blight.explain.shap_explain import attention_map, shap_values
from potato_blight.models import load_model_dir
from potato_blight.models.preprocess import preprocess

RUN_ID = "<run_id del modelo final>"
carpeta = mlflow.artifacts.download_artifacts(run_id=RUN_ID, artifact_path="model")
model, cfg = load_model_dir(carpeta)
cam = GradCAM(model, cfg["architecture"])

In [ ]:
img = Image.open("<ruta a una hoja>").convert("RGB")
x = preprocess(img, cfg)
probs, mapa = cam(x, cfg["temperature"])
dict(zip(cfg["classes"], probs.numpy().round(3))), mapa.shape

In [ ]:
# Estabilidad: correlación entre el mapa original y los mapas con ruido, brillo y desplazamiento
stability(cam, x, cfg["temperature"])

In [ ]:
# Relevancia: fracción del mapa dentro de la lesión anotada (máscara binaria del mismo encuadre)
mascara = np.array(Image.open("<ruta a la máscara>").convert("L")) > 0
relevance(mapa, mascara)

In [ ]:
# Atención por parche del modelo H2 (EfficientNet-Atención-BiLSTM)
attention_map(model, x) if cfg["architecture"] == "effnet_att_bilstm" else "Solo para H2"

In [ ]:
# SHAP (GradientExplainer) con un fondo pequeño
fondo = torch.cat([x * 0, x * 0.5])
valores = shap_values(model, fondo, x, n_samples=30)